# 03 Modelling

Reads what `python -m house_prices.train`, `evaluate` and `intervals` produced. Tuning and training happen in
those commands, not here, so the notebook stays quick and cannot touch the test set by accident. Two evaluation
setups are reported: a **random split** (shuffled 80/20, stratified by city) and a **time split** (train on older
listings, test on the newest 20%).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from house_prices import clean, config, load

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
config.PATHS.figures_dir.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(config.PATHS.figures_dir / f"{name}.png", dpi=140, bbox_inches="tight")

In [ ]:
import json

import joblib
from IPython.display import display

from house_prices import evaluate, features, formatting, split, train

metrics = json.loads(config.PATHS.metrics_json.read_text())
summaries = {s: json.loads((config.PATHS.candidates_dir / f"{s}_summary.json").read_text()) for s in split.SETUPS}
print("Synthetic data:", metrics["meta"]["synthetic_data"])
print("Deployed:", metrics["meta"]["deployed_model"], "trained with the", metrics["meta"]["deployed_setup"], "split")

## The pipeline

Preprocessing sits inside one scikit-learn `Pipeline`, so cross-validation refits every step on each training
fold. The location target encoder is the important one: fitted on the full data it would leak the price of the
rows it later predicts.

In [ ]:
from sklearn import set_config

set_config(display="text")
print(train.make_pipeline("lightgbm"))
print()
print("Inputs:", features.INPUT_COLUMNS)
print("Numeric:", features.NUMERIC_FEATURES)
print("Categorical (one-hot):", features.CATEGORICAL_FEATURES)
print(f"Location: grouped when fewer than {config.MIN_LOCATION_COUNT} training listings, then target-encoded (smoothed, 5-fold cross-fitted)")

## Tuning setup

In [ ]:
rows = []
for name in train.MODEL_NAMES:
    payload = summaries["random"]["summaries"][name]
    rows.append({"model": payload["label"], "search iterations": payload["n_iter"], "CV folds": payload["cv_folds"], "rows searched": payload["cv_rows"], "best parameters": payload["best_params"]})
display(pd.DataFrame(rows))
print("Search spaces (kind, low, high):")
for name in train.MODEL_NAMES:
    print(f"  {name}: {train.SEARCH_SPACES[name]}")

## Cross-validation on the training set only

In [ ]:
def cv_table(setup):
    rows = []
    for name in evaluate.MODEL_ORDER:
        cv = summaries[setup]["summaries"][name]["cv"]
        rows.append(
            {
                "model": evaluate.MODEL_LABELS[name],
                "RMSE (log) mean": cv["rmse_log"]["mean"],
                "RMSE (log) std": cv["rmse_log"]["std"],
                "R2 (log) mean": cv["r2_log"]["mean"],
                "MdAPE mean": cv["mdape"]["mean"],
                "MAE (PKR) mean": cv["mae_pkr"]["mean"],
            }
        )
    return pd.DataFrame(rows).round(4)


for setup in split.SETUPS:
    print(f"CV on the {setup} split's training set")
    display(cv_table(setup))

## Location encoding: target encoding against one-hot

LightGBM with the tuned parameters, cross-validated on the training set of the deployed split.

In [ ]:
ablation = metrics.get("encoding_ablation")
if ablation:
    display(pd.DataFrame({k: {m: v[m]["mean"] for m in ("rmse_log", "r2_log", "mdape", "mae_pkr")} for k, v in ablation.items()}).T.round(4))

## Test-set results, both setups

The test rows were used for the first time in `python -m house_prices.evaluate`.

In [ ]:
def test_table(setup):
    rows = []
    for name in evaluate.MODEL_ORDER:
        t = metrics["setups"][setup]["models"][name]["test"]
        rows.append(
            {
                "model": evaluate.MODEL_LABELS[name],
                "MAE": formatting.format_pkr(t["mae_pkr"]),
                "RMSE": formatting.format_pkr(t["rmse_pkr"]),
                "MdAPE": f"{t['mdape']:.1%}",
                "MAPE": f"{t['mape']:.1%}",
                "R2 (log)": round(t["r2_log"], 3),
            }
        )
    return pd.DataFrame(rows)


for setup in split.SETUPS:
    s = metrics["setups"][setup]
    print(f"{setup} split: train {s['train_rows']:,} ({s['train_date_range'][0]} to {s['train_date_range'][1]}), test {s['test_rows']:,} ({s['test_date_range'][0]} to {s['test_date_range'][1]}); chosen model {evaluate.MODEL_LABELS[s['chosen_model']]}")
    display(test_table(setup))

In [ ]:
fig = plt.figure(figsize=(13, 4.8))
from PIL import Image

image = Image.open(config.PATHS.figures_dir / "model_comparison.png")
plt.imshow(image)
plt.axis("off")
plt.show()

## Random split against time split

In [ ]:
lines = []
for name in evaluate.MODEL_ORDER:
    r = metrics["setups"]["random"]["models"][name]["test"]
    t = metrics["setups"]["time"]["models"][name]["test"]
    lines.append(f"{evaluate.MODEL_LABELS[name]}: median error {r['mdape']:.1%} on the random split, {t['mdape']:.1%} on the time split ({t['mdape'] - r['mdape']:+.1%} points).")
print("\n".join("- " + l for l in lines))
chosen_r = metrics["setups"]["random"]["chosen_model"]
chosen_t = metrics["setups"]["time"]["chosen_model"]
cv_r = metrics["setups"]["random"]["models"][chosen_r]["cv"]["rmse_log"]["mean"]
te_r = metrics["setups"]["random"]["models"][chosen_r]["test"]["rmse_log"]
te_t = metrics["setups"]["time"]["models"][chosen_t]["test"]["rmse_log"]
print()
print(f"Chosen model, RMSE on log price: CV {cv_r:.3f}, random-split test {te_r:.3f}, time-split test {te_t:.3f}.")

A shuffled split lets the model see listings from the same weeks as its test rows (the same agents, near
duplicates, the same market level). A time split asks it to price listings from a later period, which is how it
would be used. When the time-split error is larger, that gap is the honest number to quote.

## 80% prediction interval

In [ ]:
rows = []
for setup, block in metrics["intervals"].items():
    o = block["overall"]
    rows.append({"split": setup, "coverage": f"{o['coverage']:.1%}", "target": f"{block['level']:.0%}", "mean width": formatting.format_pkr(o["mean_width_pkr"]), "mean width / estimate": f"{o['mean_relative_width']:.0%}"})
display(pd.DataFrame(rows))
for setup, block in metrics["intervals"].items():
    print(f"Coverage by city, {setup} split")
    display(pd.DataFrame(block["by_city"]).assign(coverage=lambda d: (d["coverage"] * 100).round(1), mean_width_pkr=lambda d: d["mean_width_pkr"].round(0)))